In [ ]:
import pickle
import numpy as np
import pandas as pd
import streamlit as st
import joblib

In [4]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 51.4 MB/s eta 0:00:00


In [6]:
st.set_page_config(
    page_title="Product Recommendation System",
    layout="wide"
)

st.title("Product Recommendation System")
st.write("This app uses saved trained models for product recommendation.")

2026-10-07 22:12:07.151 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:12:07.156 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:12:07.537 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-07 22:12:07.538 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:12:07.542 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:12:07.545 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:12:07.550 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [8]:
import joblib

@st.cache_resource
def load_model_bundle():
    model_bundle = joblib.load("models/product_recommendation_models.joblib")
    return model_bundle

In [ ]:
import joblib
import streamlit as st

@st.cache_resource
def load_model_bundle():
    return joblib.load("models/product_recommendation_models.joblib")

model_bundle = load_model_bundle()

2026-10-08 04:07:26.811 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
C:\Users\praka\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator KMeans from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\praka\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MiniBatchKMeans from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  w

In [10]:
kmeans_model = model_bundle["kmeans_model"]
minibatch_kmeans_model = model_bundle["minibatch_kmeans_model"]
svd_model = model_bundle["svd_model"]
cosine_model = model_bundle["cosine_model"]

user_item_matrix = model_bundle["user_item_matrix"]

user_ids = model_bundle["user_ids"]
product_ids = model_bundle["product_ids"]

user_to_idx = model_bundle["user_to_idx"]
product_to_idx = model_bundle["product_to_idx"]

kmeans_user_cluster_map = model_bundle["kmeans_user_cluster_map"]
minibatch_user_cluster_map = model_bundle["minibatch_user_cluster_map"]

kmeans_cluster_top_products = model_bundle["kmeans_cluster_top_products"]
minibatch_cluster_top_products = model_bundle["minibatch_cluster_top_products"]

global_top_products = model_bundle["global_top_products"]
popular_products = model_bundle["popular_products"]
seen_by_user = model_bundle["seen_by_user"]

NameError: name 'model_bundle' is not defined

In [11]:
def recommend_for_user(user_id, model_name="KMeans", top_n=10):
    already_seen = seen_by_user.get(user_id, set())

    if model_name == "KMeans":
        user_cluster_map = kmeans_user_cluster_map
        cluster_top_products = kmeans_cluster_top_products
    else:
        user_cluster_map = minibatch_user_cluster_map
        cluster_top_products = minibatch_cluster_top_products

    if user_id in user_cluster_map:
        cluster = user_cluster_map[user_id]
        candidate_products = cluster_top_products.get(cluster, [])
    else:
        cluster = "New / Unknown User"
        candidate_products = []

    recommendations = [
        product for product in candidate_products
        if product not in already_seen
    ][:top_n]

    if len(recommendations) < top_n:
        fallback_products = [
            product for product in global_top_products
            if product not in recommendations and product not in already_seen
        ]

        recommendations = recommendations + fallback_products[:top_n - len(recommendations)]

    return cluster, recommendations

In [12]:
def recommend_similar_products(product_id, top_n=10):
    if product_id not in product_to_idx:
        return pd.DataFrame(columns=["productid", "similarity_score"])

    product_index = product_to_idx[product_id]

    n_neighbors = min(top_n + 1, len(product_ids))

    distances, indices = cosine_model.kneighbors(
        user_item_matrix.T[product_index],
        n_neighbors=n_neighbors
    )

    recommended_indices = indices.flatten()[1:]
    similarity_scores = 1 - distances.flatten()[1:]

    result = pd.DataFrame({
        "productid": product_ids[recommended_indices],
        "similarity_score": similarity_scores
    })

    return result

In [13]:
def get_popular_products(top_n=10):
    return popular_products.head(top_n)

In [14]:
st.sidebar.header("Settings")

model_choice = st.sidebar.selectbox(
    "Choose clustering model",
    ["KMeans", "MiniBatchKMeans"]
)

top_n = st.sidebar.slider(
    "Number of recommendations",
    min_value=5,
    max_value=20,
    value=10
)

2026-10-07 22:25:15.761 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:15.762 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:15.763 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:15.765 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:15.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:15.771 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:15.772 Session state does not function when running a script without `streamlit run`
2026-10-07 22:25:15.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25

In [15]:
tab1, tab2, tab3 = st.tabs([
    "User Recommendations",
    "Similar Products",
    "Popular Products"
])

2026-10-07 22:25:24.789 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:24.790 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:24.791 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:24.791 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [16]:
with tab1:
    st.subheader("User-Based Product Recommendations")

    user_id_input = st.text_input(
        "Enter User ID",
        value=str(user_ids[0])
    )

    if st.button("Recommend Products for User"):
        cluster, recommendations = recommend_for_user(
            user_id_input,
            model_name=model_choice,
            top_n=top_n
        )

        st.write("Selected Model:", model_choice)
        st.write("User Cluster:", cluster)

        recommendation_df = pd.DataFrame({
            "recommended_productid": recommendations
        })

        st.dataframe(recommendation_df, use_container_width=True)

2026-10-07 22:25:34.056 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:34.057 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:34.058 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name 'user_ids' is not defined

In [17]:
with tab2:
    st.subheader("Cosine Similarity Product Recommendations")

    product_id_input = st.text_input(
        "Enter Product ID",
        value=str(product_ids[0])
    )

    if st.button("Find Similar Products"):
        similar_products = recommend_similar_products(
            product_id_input,
            top_n=top_n
        )

        if similar_products.empty:
            st.warning("Product ID not found.")
        else:
            st.dataframe(similar_products, use_container_width=True)

2026-10-07 22:25:44.602 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:44.605 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:44.610 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name 'product_ids' is not defined

In [18]:
with tab3:
    st.subheader("Popularity-Based Recommendations")

    top_popular_products = get_popular_products(top_n)

    st.dataframe(top_popular_products, use_container_width=True)

2026-10-07 22:25:54.873 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:54.875 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-07 22:25:54.876 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name 'popular_products' is not defined